In [28]:
!pip -q install PyMuPDF pillow transformers accelerate qwen-vl-utils

In [29]:
import io
import os
import re
import json
import time
from pathlib import Path
from typing import List, Optional

import fitz
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

In [30]:
CONTRACT_EXTRACTION_PROMPT = """
You are an advanced OCR system specialized in Arabic and English legal contracts.

مهمتك الوحيدة:
Extract ALL text exactly as shown in the image with maximum accuracy.

Rules / القواعد:

1. Extract everything exactly:
- Arabic text
- English text
- Numbers
- Dates
- Names
- Addresses
- Currency amounts
- Article numbers
- Tables

2. Formatting:
- Keep original reading order
- Arabic from right to left
- Preserve paragraphs
- Put titles in separate lines
- Tables as:
column1 | column2 | column3

3. Never:
- Do not explain
- Do not summarize
- Do not rewrite
- Do not translate
- Do not add comments

4. If unclear:
Use [غير واضح] or [unclear]

Start extraction now.
"""


# CONTRACT_TYPE_DETECTION_PROMPT = """بناءً على النص التالي من عقد، حدد نوعه بكلمة واحدة فقط من الخيارات التالية:
# - employment (عقد عمل)
# - rental (عقد إيجار)
# - nda (اتفاقية سرية)
# - other (غير ذلك)

# النص:
# {text}

# النوع:"""


MERGE_PAGES_PROMPT = """لديك نص مستخرج من {page_count} صفحات لعقد واحد.
دورك: ادمج هذه الصفحات في نص متسق واحد مع:
- الحفاظ على ترتيب الصفحات
- إزالة الترويسات والتذييلات المتكررة (أرقام الصفحات، اسم الشركة المتكرر)
- الاحتفاظ بكل المحتوى الجوهري
- لا تضيف ولا تحذف من المحتوى الأصلي

الصفحات:
{pages_text}

النص المدمج:"""

In [31]:
DEFAULT_DPI = 200


def pdf_to_images(file_bytes: bytes, dpi: int = DEFAULT_DPI) -> List[Image.Image]:
    images = []
    doc = fitz.open(stream=file_bytes, filetype="pdf")

    zoom = dpi / 72
    matrix = fitz.Matrix(zoom, zoom)

    for page_num in range(len(doc)):
        page = doc[page_num]
        pixmap = page.get_pixmap(matrix=matrix, alpha=False)
        img_bytes = pixmap.tobytes("png")
        pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        images.append(pil_img)

    doc.close()
    return images


def load_image(file_bytes: bytes, filename: str) -> List[Image.Image]:
    img = Image.open(io.BytesIO(file_bytes)).convert("RGB")
    return [img]


def file_to_images(file_bytes: bytes, filename: str, dpi: int = DEFAULT_DPI) -> dict:
    ext = Path(filename).suffix.lower()

    try:
        if ext == ".pdf":
            images = pdf_to_images(file_bytes, dpi)
            return {
                "images": images,
                "page_count": len(images),
                "file_type": "pdf",
                "success": True,
                "error": None
            }

        elif ext in (".png", ".jpg", ".jpeg", ".webp", ".tiff", ".tif", ".bmp"):
            images = load_image(file_bytes, filename)
            return {
                "images": images,
                "page_count": 1,
                "file_type": "image",
                "success": True,
                "error": None
            }

        else:
            return {
                "images": [],
                "page_count": 0,
                "file_type": "unsupported",
                "success": False,
                "error": f"نوع الملف غير مدعوم: {ext}. يُقبل فقط PDF أو صور"
            }

    except Exception as e:
        return {
            "images": [],
            "page_count": 0,
            "file_type": ext.replace(".", ""),
            "success": False,
            "error": f"خطأ في معالجة الملف: {str(e)}"
        }


def resize_for_vlm(img: Image.Image, max_pixels: int = 1280 * 1280) -> Image.Image:
    w, h = img.size
    current_pixels = w * h

    if current_pixels > max_pixels:
        scale = (max_pixels / current_pixels) ** 0.5
        new_w = int(w * scale)
        new_h = int(h * scale)
        img = img.resize((new_w, new_h), Image.LANCZOS)

    return img


def clean_extracted_text(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    lines = [line.strip() for line in text.splitlines()]
    return "\n".join(lines).strip()

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
USE_4BIT = False 

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print(f"\n⏳ Loading model: {MODEL_ID}")

if USE_4BIT:
    from transformers import BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )

processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Model loaded successfully.")

GPU available: True
GPU: Tesla T4

⏳ Loading model: Qwen/Qwen2.5-VL-3B-Instruct


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Model loaded successfully.


In [33]:
def extract_page(img: Image.Image) -> str:
    img = resize_for_vlm(img)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": CONTRACT_EXTRACTION_PROMPT}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=4096,
            temperature=0.0,
            do_sample=False,
            repetition_penalty=1.05,
        )

    generated = output_ids[:, inputs.input_ids.shape[1]:]
    result = processor.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return result[0]

In [ ]:
class ContractVLMExtractor:
    def extract_from_images(self, images: List[Image.Image], verbose: bool = True) -> dict:
        if not images:
            return {
                "full_text": "",
                "pages": [],
                "page_count": 0,
                "total_time": 0,
                "success": False,
                "error": "لا توجد صور للمعالجة"
            }

        pages_results = []
        start_total = time.time()

        for i, img in enumerate(images):
            page_num = i + 1

            if verbose:
                print(f"معالجة صفحة {page_num}/{len(images)}...", end=" ", flush=True)

            start = time.time()

            try:
                text = extract_page(img)
                text = clean_extracted_text(text)
                elapsed = round(time.time() - start, 2)
                word_count = len(text.split())

                if verbose:
                    print(f" ({word_count} كلمة، {elapsed}ث)")

                pages_results.append({
                    "page": page_num,
                    "text": text,
                    "word_count": word_count,
                    "time": elapsed
                })

            except Exception as e:
                elapsed = round(time.time() - start, 2)

                if verbose:
                    print(f" خطأ: {e}")

                pages_results.append({
                    "page": page_num,
                    "text": f"[خطأ في استخراج الصفحة {page_num}: {str(e)}]",
                    "word_count": 0,
                    "time": elapsed
                })

        total_time = round(time.time() - start_total, 2)
        full_text = self._merge_pages(pages_results)
        total_words = len(full_text.split())

        if verbose:
            print(f"\n اكتمل الاستخراج — {len(images)} صفحة | {total_words} كلمة | {total_time}ث")

        return {
            "full_text": full_text,
            "pages": pages_results,
            "page_count": len(images),
            "total_words": total_words,
            "total_time": total_time,
            "success": bool(full_text.strip()),
            "error": None
        }

    def _merge_pages(self, pages_results: list) -> str:
        if len(pages_results) == 1:
            return pages_results[0]["text"]

        parts = []
        for p in pages_results:
            parts.append(f"{'─' * 40}")
            parts.append(f"[ صفحة {p['page']} ]")
            parts.append(f"{'─' * 40}")
            parts.append(p["text"])
            parts.append("")

        return "\n".join(parts).strip()

In [ ]:
def extract_contract(
    file_path: str,
    dpi: int = 200,
    output_json: bool = True,
) -> dict:
    file_path = Path(file_path)

    if not file_path.exists():
        return {"success": False, "error": f"الملف غير موجود: {file_path}"}

    file_bytes = file_path.read_bytes()

    print("⏳ تحويل الملف إلى صور...")
    images_result = file_to_images(file_bytes, file_path.name, dpi=dpi)

    if not images_result["success"]:
        return {"success": False, "error": images_result["error"]}

    print(f" {images_result['page_count']} صفحة/صورة جاهزة للمعالجة")

    extractor = ContractVLMExtractor()

    print(" استخراج النص...\n")
    extraction = extractor.extract_from_images(images_result["images"], verbose=True)

    if not extraction["success"]:
        return {"success": False, "error": extraction["error"]}

    output_data = {
        "file": str(file_path),
        "file_type": images_result["file_type"],
        "page_count": extraction["page_count"],
        "total_words": extraction["total_words"],
        "total_time_seconds": extraction["total_time"],
        "model": MODEL_ID,
        "full_text": extraction["full_text"],
        "pages": extraction["pages"],
        "success": True,
    }

    if output_json:
        out_path = file_path.with_suffix(".extracted.json")
        out_path.write_text(json.dumps(output_data, ensure_ascii=False, indent=2))
        print(f"\n النتيجة محفوظة: {out_path}")

    return output_data

In [38]:
from google.colab import files

uploaded = files.upload()

Saving CNN .pdf to CNN .pdf


In [ ]:
filename = list(uploaded.keys())[0]
import torch
torch.cuda.empty_cache()

result = extract_contract(filename, dpi=200)

print("\n" + "=" * 60)
print("النص المستخرج:")
print("=" * 60)
print(result["full_text"][:5000])

In [ ]:
from google.colab import files

json_name = Path(filename).with_suffix(".extracted.json").name
files.download(json_name)